![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 18 -- Lab 2: Build a Chatbot with Gemma-4

**Scenario:** A tech startup wants to build an AI assistant that can answer questions, adopt different personalities, and even chat in Arabic. Your job is to build a chatbot using Google's **Gemma-4** model, learning how **system prompts**, **user prompts**, and **generation parameters** (temperature, top-k, top-p) shape the model's output.

This connects directly to today's slides on text generation strategies: you'll see temperature, top-k, and top-p in action!

| Part | Goal |
|---|---|
| Part 1 | Setup: install libraries and load Gemma-4 |
| Part 2 | First generation: your first chat with Gemma |
| Part 3 | System vs user prompts: the chat template |
| Part 4 | Temperature: the creativity dial |
| Part 5 | Top-k and top-p: controlling randomness |
| Part 6 | System prompt personalities |
| Part 7 | Multi-turn conversations |
| Part 8 | (Bonus) Arabic chatbot |

---

## Setup

In [ ]:
!pip install --upgrade transformers accelerate -q

---

## Part 1: Load Gemma-4

We use **Gemma-4 E2B-it** -- Google's smallest instruction-tuned Gemma 4 model:
- 2.3B effective parameters (fits easily on a T4 GPU)
- Instruction-tuned: trained to follow instructions and have conversations
- Supports system prompts, multi-turn chat, and 140+ languages
- The `-it` suffix means "instruction-tuned" (as opposed to the base model)

In [ ]:
# --- GIVEN ---
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

print(f"Loading {MODEL_ID}...")
print("This may take a minute on first run (downloading the model).")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model loaded on: {model.device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---

## Part 2: Your First Chat with Gemma

Chat models expect messages in a specific format: a list of dictionaries with `role` and `content`. The roles are:
- **system**: Instructions for how the model should behave (invisible to the "user")
- **user**: What the human says
- **assistant**: What the model responds (we leave this for the model to generate)

The `processor.apply_chat_template()` converts this list into the special format Gemma expects.

In [ ]:
# --- GIVEN ---
from transformers import TextStreamer

def chat(messages, max_new_tokens=256, temperature=1.0, top_k=50, top_p=0.95,
         do_sample=True, stream=False):
    """Send messages to Gemma and return the response.
    Set stream=True to watch tokens appear live (next-token prediction in action!).
    """
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    streamer = TextStreamer(processor, skip_prompt=True,
                           skip_special_tokens=True) if stream else None

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            streamer=streamer,
        )

    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

In [ ]:
# --- GIVEN ---
# Our first conversation!
messages = [
    {"role": "user", "content": "What is the capital of Saudi Arabia?"}
]

response = chat(messages, do_sample=False)
print(f"User: {messages[0]['content']}")
print(f"Gemma: {response}")

### Streaming: Watch Next-Token Prediction Live

Remember from the slides: decoder models generate text **one token at a time**. With `stream=True`, you can watch this happen in real time -- each word appears as the model predicts it, just like ChatGPT!

In [ ]:
# --- GIVEN ---
messages = [
    {"role": "user", "content": "Explain how a rocket works in 3 sentences."}
]

print("Gemma (streaming): ", end="")
_ = chat(messages, stream=True, do_sample=False)

Let's see what the raw chat template looks like:

In [ ]:
# --- GIVEN ---
raw_template = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("Raw chat template:")
print(repr(raw_template))

---

## Part 3: System Prompts

A **system prompt** tells the model how to behave -- its personality, rules, and constraints. The user never sees the system prompt, but it shapes every response.

### Task 1: Add a System Prompt

**TODO:**
- Create a message list with a `system` role message and a `user` message
- Try the system prompt: "You are a helpful science tutor for middle school students. Explain things simply using everyday examples."
- Ask: "What is gravity?"
- Then change the system prompt to: "You are a strict professor who only answers in exactly one sentence."
- Ask the same question and compare the two responses

In [ ]:
# Your code here


---

## Part 4: Temperature -- The Creativity Dial

From today's slides:
- **Low temperature** (0.1-0.3): focused, deterministic, repetitive
- **Medium temperature** (0.5-0.8): balanced
- **High temperature** (1.0+): creative, surprising, sometimes nonsensical

### Task 2: Temperature Experiment

**TODO:**
- Use the same prompt for all runs: "Tell me a short story about a cat who learns to code."
- Generate responses at temperatures: 0.1, 0.5, 0.7, 1.0, 1.5
- Print each response with its temperature
- Observe: How does the output change as temperature increases?

In [ ]:
# Your code here


### Task 3: Temperature for Different Tasks

**TODO:** Try low temperature (0.1) and high temperature (1.5) for each of these tasks. Which temperature works better for each?

1. "What is 2 + 2?" (factual math)
2. "Write a haiku about the moon." (creative poetry)
3. "Translate 'hello world' to French." (translation)

In [ ]:
# Your code here


---

## Part 5: Top-k and Top-p

From today's slides:
- **Top-k**: Only consider the top k most likely words (cuts off the tail)
- **Top-p** (nucleus): Keep the smallest set of words whose probabilities sum to p

### Task 4: Top-k Experiment

**TODO:**
- Use the prompt: "Once upon a time in a magical kingdom,"
- Generate with top_k values: 5, 20, 50, 200
- Keep temperature=0.8 and top_p=1.0 (disabled) for all
- How does the variety change?

In [ ]:
# Your code here


### Task 5: Top-p Experiment

**TODO:**
- Same prompt: "Once upon a time in a magical kingdom,"
- Generate with top_p values: 0.5, 0.8, 0.92, 0.99
- Keep temperature=0.8 and top_k=0 (disabled) for all
- Compare: how is top-p different from top-k?

In [ ]:
# Your code here


---

## Part 6: System Prompt Personalities

### Task 6: Give Gemma Different Personalities

**TODO:**
- Create 4 different system prompts that give the model distinct personalities:
  1. A **pirate** who talks like a pirate
  2. A **scientist** who always cites evidence
  3. A **poet** who answers in rhyming verse
  4. A **strict teacher** who always quizzes the student back
- Ask all four the same question: "Why is the sky blue?"
- Print each personality's response

In [ ]:
# Your code here


---

## Part 7: Multi-Turn Conversations

Real chatbots remember what you said earlier. To do this, we keep a list of all messages and add each new user message and assistant response to the list before sending the next message.

### Task 7: Build a Multi-Turn Chat

**TODO:**
1. Start with a system prompt: "You are a friendly AI tutor. Keep your answers short and clear."
2. Send: "What is machine learning?"
3. Get the response, append it to the message history as an `assistant` message
4. Send a follow-up: "Can you give me a simple example?"
5. Get the response and append it
6. Send another follow-up: "How is that different from regular programming?"
7. Print the entire conversation

In [ ]:
# Your code here


### Task 8: Interactive Chat Loop

**TODO:** Build an interactive chat loop that:
1. Starts with a system prompt of your choice
2. Reads user input with `input()`
3. Generates a response with `stream=True` so tokens appear live
4. Appends both user and assistant messages to history
5. Loops until the user types "exit"

In [ ]:
# Your code here


---

## Part 8: (Bonus) Arabic Chatbot

Gemma-4 supports 140+ languages including Arabic. Let's test it!

### Task 9: Arabic System Prompt

**TODO:**
- Set the system prompt in Arabic: "انت مساعد ذكي ودود. اجب دائما باللغة العربية بشكل مبسط ومختصر."
- Ask in Arabic: "ما هو الذكاء الاصطناعي؟"
- Then ask: "كيف يمكنني تعلم البرمجة؟"
- Does the model respond in Arabic? How good is the quality?

In [ ]:
# Your code here


---

## Discussion

1. How did the system prompt change the model's behavior? Was it always consistent with the personality you set?
2. What temperature worked best for factual questions vs creative writing? Why?
3. Did you notice a difference between top-k and top-p? Which felt more natural?
4. In the multi-turn conversation, did the model remember context from earlier messages? Give an example.
5. How well did Gemma handle Arabic compared to English? Any differences in quality?

---

## Wrap-Up

**What you learned:**
- How to load and use an instruction-tuned LLM (Gemma-4) for text generation
- The chat template format: system, user, and assistant roles
- System prompts control the model's personality and behavior
- Temperature controls creativity vs determinism
- Top-k and top-p filter unlikely tokens to improve quality
- Multi-turn conversations by maintaining message history
- Modern LLMs are multilingual -- the same model works in Arabic and English

**Connections to today's slides:**
- Temperature, top-k, and top-p are the exact generation strategies we learned about
- Gemma-4 is a decoder-only model (like GPT) that generates text autoregressively
- The chat template is how we format the input for instruction-tuned models